# 06 — Pipeline End-to-End (Skenario 3)

Notebook ini menjalankan seluruh tahap analisis secara berurutan pada **dataset besar** (10.000 pelanggan, 1 juta order).
Setiap tahap membaca output tahap sebelumnya dari MinIO — mendemonstrasikan konsep **DAG pipeline** yang menjadi dasar Apache Airflow.

```
raw/large/  →  Preprocessing  →  Clustering  →  Classification  →  Association Rules
```

## Prasyarat
Pastikan dataset besar sudah ada di MinIO sebelum menjalankan notebook ini:
```bash
# Di terminal lokal (bukan di dalam Docker)
python -m data.generate_large_dataset
```

## 1. Setup

In [ ]:
import os, time
from dotenv import load_dotenv
load_dotenv()

import sys
sys.path.insert(0, "/home/jovyan/work")

from analysis.spark_session import create_spark_session
spark = create_spark_session("06 - Pipeline End-to-End")

BUCKET = "datalake"
timings: dict[str, float] = {}
print(f"Spark master: {spark.sparkContext.master}")
print(f"Worker count: {spark.sparkContext.defaultParallelism}")

## 2. Load Dataset Besar dari MinIO

In [ ]:
t0 = time.time()

df_customers = spark.read.csv(
    f"s3a://{BUCKET}/raw/large/customers/", header=True, inferSchema=True
)
df_products = spark.read.csv(
    f"s3a://{BUCKET}/raw/large/products/", header=True, inferSchema=True
)
df_orders = spark.read.csv(
    f"s3a://{BUCKET}/raw/large/orders/", header=True, inferSchema=True
)

# Cache karena akan dipakai berkali-kali
df_customers.cache()
df_products.cache()
df_orders.cache()

n_customers = df_customers.count()
n_products  = df_products.count()
n_orders    = df_orders.count()

timings["load"] = time.time() - t0
print(f"Pelanggan : {n_customers:,}")
print(f"Produk    : {n_products:,}")
print(f"Order     : {n_orders:,}")
print(f"Waktu load: {timings['load']:.1f} detik")

## 3. Preprocessing — Feature Engineering

In [ ]:
from pyspark.sql import functions as F

t0 = time.time()

# Join orders + products untuk hitung total spend
df_order_detail = df_orders.join(df_products.select("product_id", "price"), on="product_id")
df_order_detail = df_order_detail.withColumn("spend", F.col("quantity") * F.col("price"))

df_agg = df_order_detail.groupBy("customer_id").agg(
    F.count("order_id").alias("total_orders"),
    F.sum("spend").alias("total_spend"),
    F.avg("spend").alias("avg_spend_per_order"),
)

df_features = df_customers.join(df_agg, on="customer_id", how="left").fillna(0)

# Label is_high_value: 1 jika total_spend >= median
median_spend = df_features.approxQuantile("total_spend", [0.5], 0.01)[0]
df_features = df_features.withColumn(
    "is_high_value", F.when(F.col("total_spend") >= median_spend, 1).otherwise(0)
)

# Simpan ke processed zone
df_features.write.mode("overwrite").option("header", True).csv(
    f"s3a://{BUCKET}/processed/large/customer_features/"
)

timings["preprocessing"] = time.time() - t0
print(f"Fitur dibangun untuk {df_features.count():,} pelanggan")
print(f"Waktu preprocessing: {timings['preprocessing']:.1f} detik")
df_features.show(5)

## 4. Clustering — K-Means

In [ ]:
from analysis.mining.clustering import run_kmeans

t0 = time.time()

df_features_cached = spark.read.csv(
    f"s3a://{BUCKET}/processed/large/customer_features/", header=True, inferSchema=True
)

df_clustered, model = run_kmeans(
    df_features_cached,
    feature_cols=["age", "total_orders", "total_spend"],
    k=5,
)

df_clustered.drop("features").write.mode("overwrite").option("header", True).csv(
    f"s3a://{BUCKET}/processed/large/customer_clusters/"
)

timings["clustering"] = time.time() - t0
print(f"Waktu clustering: {timings['clustering']:.1f} detik")
print()

df_clustered.groupBy("cluster").agg(
    F.count("customer_id").alias("jumlah"),
    F.round(F.avg("age"), 1).alias("rata_usia"),
    F.round(F.avg("total_orders"), 1).alias("rata_orders"),
    F.round(F.avg("total_spend"), 0).alias("rata_spend"),
).orderBy("cluster").show()

## 5. Classification — Decision Tree

In [ ]:
from analysis.mining.classification import run_decision_tree

t0 = time.time()

df_preds, auc = run_decision_tree(
    df_features_cached,
    numeric_cols=["age", "total_orders", "total_spend", "avg_spend_per_order"],
    categorical_cols=["city"],
    label_col="is_high_value",
)

cols_to_drop = ["rawPrediction", "probability"]
df_to_save = df_preds.drop(*[c for c in cols_to_drop if c in df_preds.columns])
df_to_save.write.mode("overwrite").option("header", True).csv(
    f"s3a://{BUCKET}/processed/large/classification_results/"
)

timings["classification"] = time.time() - t0
print(f"AUC: {auc:.4f}")
print(f"Waktu classification: {timings['classification']:.1f} detik")

## 6. Association Rules — FP-Growth

In [ ]:
from analysis.mining.association import run_fpgrowth

t0 = time.time()

# Bangun format transaksi: (customer_id, items: list[product_name])
df_tx = df_orders.join(df_products.select("product_id", "name"), on="product_id") \
    .groupBy("customer_id") \
    .agg(F.collect_set("name").alias("items"))

df_freq, df_rules = run_fpgrowth(df_tx, min_support=0.05, min_confidence=0.3)

# Konversi Array → String sebelum simpan ke CSV
df_rules_csv = df_rules \
    .withColumn("antecedent", F.concat_ws(", ", F.col("antecedent"))) \
    .withColumn("consequent", F.concat_ws(", ", F.col("consequent")))

df_rules_csv.write.mode("overwrite").option("header", True).csv(
    f"s3a://{BUCKET}/processed/large/association_rules/"
)

timings["association"] = time.time() - t0
print(f"Jumlah frequent itemsets : {df_freq.count()}")
print(f"Jumlah rules             : {df_rules.count()}")
print(f"Waktu association rules  : {timings['association']:.1f} detik")
print()
df_rules.orderBy("lift", ascending=False).show(10, truncate=False)

## 7. Ringkasan Waktu Eksekusi Pipeline

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df_timing = pd.DataFrame([
    {"Tahap": k, "Waktu (detik)": round(v, 1)}
    for k, v in timings.items()
])
df_timing["Kumulatif (detik)"] = df_timing["Waktu (detik)"].cumsum().round(1)
print(df_timing.to_string(index=False))
print(f"\nTotal: {sum(timings.values()):.1f} detik")

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(df_timing["Tahap"], df_timing["Waktu (detik)"], color="steelblue")
ax.set_xlabel("Waktu (detik)")
ax.set_title(f"Waktu Eksekusi Pipeline — {spark.sparkContext.master}")
for i, v in enumerate(df_timing["Waktu (detik)"]):
    ax.text(v + 0.5, i, f"{v:.1f}s", va="center")
plt.tight_layout()
plt.show()